# Deep Boltzmann Machine: 多層のエネルギーモデル

DBMはRBMを多層に拡張したエネルギーベースモデルである。表現力は上がるが、隠れ層同士が依存し推論が難しくなる。


## このノートの読み方

想定読者: RBM、エネルギー関数、sigmoid、近似の必要性を理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

RBMでは`p(h|v)`を簡単に計算できた。DBMでは隠れ層が複数あるため、平均場近似で確率的な平均`mu`を反復更新する。


## 到達目標

- DBMとRBM/DBNの違いを説明できる
- 平均場近似の更新を追える
- Trainer例が概念デモである限界を説明できる


## 重要語句

- `mean field`: 隠れ変数を平均確率で近似する方法
- `pretraining`: 層ごとにRBMとして初期化する歴史的工夫
- `DBN`: 向き付き/無向が混ざる別モデル


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| v | (B, n_visible) | 可視層 |
| mu1 | (B, n_h1) | 第一隠れ層の平均 |
| mu2 | (B, n_h2) | 第二隠れ層の平均 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| DBM学習の注意 | DBM本体の厳密学習は分配関数と隠れ変数推論が難しい。このノートのTrainer例は平均場CDサロゲートであり、実用DBM実装ではない。 |
| 完全なenergy | `E=-b^Tv-c_1^Th_1-c_2^Th_2-v^TW_1h_1-h_1^TW_2h_2`として、各層のbiasと隣接層の相互作用を分けて読む。 |
| 平均場 | `mu1, mu2`は0/1サンプルではなく、各ユニットが1になる確率の平均である。互いに依存するため反復更新が必要になる。 |
| RBM/DBNとの差分 | RBMは1隠れ層の無向モデル、DBMは多層無向モデル、DBNは向き付き/無向が混ざる歴史的モデルである。 |
| バイオ用途 | 多層の潜在因子で遺伝子発現や表現型の共起を説明する、近似推論教材として位置づける。 |


## Deep energy

隣接層の相互作用を足し合わせる。

$$
E(v,h^{(1)},h^{(2)})=-v^\top W^{(1)}h^{(1)}-{h^{(1)}}^\top W^{(2)}h^{(2)}-\mathrm{biases}
$$


## Mean-field update

サンプルではなく、1になる確率の平均を反復更新する。

$$
\mu^{(1)}\leftarrow\sigma(b_1+vW^{(1)}+\mu^{(2)}{W^{(2)}}^\top)
$$


## Position today

現代の主役ではないが、近似推論とEBM理解に重要である。

$$
\mathrm{expressive\ power}\uparrow,\quad \mathrm{inference\ cost}\uparrow
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
v = torch.bernoulli(torch.full((2, 4), 0.5))
W1 = torch.randn(4, 3)
W2 = torch.randn(3, 2)
mu2 = torch.full((2, 2), 0.5)
for i in range(3):
    mu1 = torch.sigmoid(v @ W1 + mu2 @ W2.T)
    mu2 = torch.sigmoid(mu1 @ W2)
    print(i, "mu1:", mu1.round(decimals=3), "mu2:", mu2.round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/deep-boltzmann-machine_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/deep-boltzmann-machine_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/deep-boltzmann-machine_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Deep Boltzmann Machine: 多層のエネルギーモデル difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### RBM vs DBM structure animation

- 学習目標: 層が増えて依存が増える
- 誤解の防止: RBMを積むだけと思う

対応する式:

$$
v-h^{(1)}-h^{(2)}
$$


<p><a href="../demos/deep-boltzmann-machine_structure.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/deep-boltzmann-machine_structure.html</code>）</p>
<iframe
  src="../demos/deep-boltzmann-machine_structure.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="RBM vs DBM structure animation"
></iframe>


### mean-field iteration animation

- 学習目標: mu1,mu2が反復で変わる
- 誤解の防止: muを0/1サンプルと思う

対応する式:

$$
\mu\leftarrow\sigma(\cdots)
$$


<p><a href="../demos/deep-boltzmann-machine_mean_field.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/deep-boltzmann-machine_mean_field.html</code>）</p>
<iframe
  src="../demos/deep-boltzmann-machine_mean_field.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="mean-field iteration animation"
></iframe>


### pretraining stack animation

- 学習目標: RBM事前学習を積み上げる
- 誤解の防止: 最初から全体を学ぶと思う

対応する式:

$$
RBM_1\to RBM_2\to DBM
$$


<p><a href="../demos/deep-boltzmann-machine_pretraining.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/deep-boltzmann-machine_pretraining.html</code>）</p>
<iframe
  src="../demos/deep-boltzmann-machine_pretraining.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="pretraining stack animation"
></iframe>


### inference difficulty animation

- 学習目標: 隠れ状態数が増える
- 誤解の防止: 深くしても推論は同じと思う

対応する式:

$$
2^{n_h}
$$


<p><a href="../demos/deep-boltzmann-machine_difficulty.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/deep-boltzmann-machine_difficulty.html</code>）</p>
<iframe
  src="../demos/deep-boltzmann-machine_difficulty.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="inference difficulty animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`Deep Boltzmann Machine: 多層のエネルギーモデル`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyDBMDataset(Dataset):
    def __init__(self, n_samples: int = 40, n_visible: int = 6) -> None:
        self.v = torch.bernoulli(torch.full((n_samples, n_visible), 0.5))

    def __len__(self) -> int:
        return len(self.v)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"v": self.v[index]}


class TrainerDBMMeanFieldCD(nn.Module):
    def __init__(self, n_visible: int = 6, n_h1: int = 4, n_h2: int = 3) -> None:
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(n_visible, n_h1) * 0.05)
        self.W2 = nn.Parameter(torch.randn(n_h1, n_h2) * 0.05)
        self.b = nn.Parameter(torch.zeros(n_visible))
        self.c1 = nn.Parameter(torch.zeros(n_h1))
        self.c2 = nn.Parameter(torch.zeros(n_h2))

    def mean_field(self, v: torch.Tensor, steps: int = 3) -> tuple[torch.Tensor, torch.Tensor]:
        mu2 = torch.full((v.shape[0], self.W2.shape[1]), 0.5, device=v.device)
        mu1 = torch.full((v.shape[0], self.W1.shape[1]), 0.5, device=v.device)
        for _ in range(steps):
            mu1 = torch.sigmoid(v @ self.W1 + mu2 @ self.W2.T + self.c1)
            mu2 = torch.sigmoid(mu1 @ self.W2 + self.c2)
        return mu1, mu2

    def energy_mean_field(self, v: torch.Tensor, mu1: torch.Tensor, mu2: torch.Tensor) -> torch.Tensor:
        visible = v @ self.b
        hidden = mu1 @ self.c1 + mu2 @ self.c2
        pair1 = torch.sum((v @ self.W1) * mu1, dim=1)
        pair2 = torch.sum((mu1 @ self.W2) * mu2, dim=1)
        return -(visible + hidden + pair1 + pair2)

    def forward(self, v: torch.Tensor) -> dict[str, torch.Tensor]:
        mu1_pos, mu2_pos = self.mean_field(v)
        with torch.no_grad():
            v_neg = torch.bernoulli(torch.sigmoid(mu1_pos @ self.W1.T + self.b))
        mu1_neg, mu2_neg = self.mean_field(v_neg)
        pos_energy = self.energy_mean_field(v, mu1_pos, mu2_pos)
        neg_energy = self.energy_mean_field(v_neg, mu1_neg, mu2_neg)
        loss = torch.mean(pos_energy - neg_energy)
        return {"loss": loss, "logits": mu1_pos, "mu2": mu2_pos}


training_args = TrainingArguments(
    output_dir="./results/deep-boltzmann-machine_trainer_demo",
    max_steps=3,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = Trainer(model=TrainerDBMMeanFieldCD(), args=training_args, train_dataset=TinyDBMDataset())
train_output = trainer.train()
print("DBM mean-field CD surrogate loss:", train_output.training_loss)


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- 平均場更新を反復回数で比較する
- RBM/DBM/DBNを図で整理する
- 変分推論とEBMへ進む


## 確認問題

- 平均場近似の`mu`は0/1値か。
- DBMがRBMより推論しにくい理由を書く。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
